In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!cp /content/drive/My\ Drive/L7andL8/thecarconnectionpicturedataset.rar /content/

In [ ]:
!pip install patool

In [ ]:
!mkdir /content/CarImages/

In [ ]:
import patoolib
patoolib.extract_archive('/content/thecarconnectionpicturedataset.rar', outdir='/content/CarImages/')

# Create an array of unstructured data and analyze it WITHOUT explicitly creating a table first.  

In [ ]:
import os
arr = os.listdir('/content/CarImages/')
print(arr[:10])

In [ ]:
# 'Make', 'Model', 'Year', 'MSRP', 'Front Wheel Size (in)', 'SAE Net Horsepower @ RPM', 'Displacement', 'Engine Type', 'Width, Max w/o mirrors (in)', 'Height, Overall (in)', 'Length, Overall (in)', 'Gas Mileage', 'Drivetrain', 'Passenger Capacity', 'Passenger Doors', 'Body Style'

In [ ]:
# Exploring the filename strings after parsing out just the first word of the filename - which represents the brand of the vehicle. 

import pandas as pd
df = pd.DataFrame({'filename': arr})
# df['filename'].str.split('_').str[0].str.split('_').str[-1]  

# How many images of each car type do we have?         
df['filename'].str.split('_').str[2].value_counts()

In [ ]:
DataFrameOfCarData = pd.DataFrame(df['filename'].str.split('_').tolist(), index= df.index)
DataFrameOfCarData.head()

In [ ]:
DataFrameOfCarData.columns = ['Make', 'Model', 'Year', 'MSRP', 'Front Wheel Size (in)', 'SAE Net Horsepower @ RPM', 'Displacement', 'Engine Type', 'Width, Max w/o mirrors (in)', 'Height, Overall (in)', 'Length, Overall (in)', 'Gas Mileage', 'Drivetrain', 'Passenger Capacity', 'Passenger Doors', 'Body Style','ID']
DataFrameOfCarData['ID'] = DataFrameOfCarData['ID'].str.split('.').str[0] 
DataFrameOfCarData.head()

In [ ]:
# DataFrameOfCarData['MSRP'] = DataFrameOfCarData['MSRP'].astype(float)

convert_dict = { 'Year':int, 'MSRP':float, 'Front Wheel Size (in)':float, 'SAE Net Horsepower @ RPM':float, 'Displacement':float, 
                'Engine Type':float, 'Width, Max w/o mirrors (in)':float, 
                'Height, Overall (in)':float, 'Length, Overall (in)':float, 'Gas Mileage':float, 
                 'Passenger Capacity':int, 'Passenger Doors':int }

DataFrameOfCarData = DataFrameOfCarData.astype(convert_dict, errors='ignore')                 

In [ ]:
DataFrameOfCarData.info()

In [ ]:
# DataFrameOfCarData['SAE Net Horsepower @ RPM'].value_counts()
# DataFrameOfCarData['SAE Net Horsepower @ RPM'] = float(DataFrameOfCarData['SAE Net Horsepower @ RPM'])

In [ ]:
DataFrameOfCarData.to_csv("DataFrameOfCarData_toProcessinR.csv")  # Export data for cleansing in R

In [ ]:
!cp  /content/DataFrameOfCarData_toProcessinR.csv  /content/drive/My\ Drive/L7andL8/

In [ ]:
!pip install seaborn

In [ ]:
import seaborn as sns
import numpy as np

df = DataFrameOfCarData.pivot_table(index='Body Style', columns='Drivetrain', values='MSRP', aggfunc=np.median)
sns.heatmap(df, annot=True, fmt=".1f")

# Analyze the cleaned data exported from R

In [ ]:
DataFrameOfCarData = pd.read_csv("/content/DataFrameOfCarData_toProcessedinR.csv")

In [ ]:
DataFrameOfCarData.head()

In [ ]:
from scipy.stats import ttest_ind as tind

# Separating two samples into different data frames
df1 = DataFrameOfCarData[DataFrameOfCarData.Drivetrain == 'AWD']
df2 = DataFrameOfCarData[DataFrameOfCarData.Drivetrain == 'FWD']

tind(df1.MSRP.dropna(), df2.MSRP.dropna(),  equal_var=False)      

# Classify the data using Libra

In [ ]:
!pip install libra

In [ ]:
DataFrameOfCarData.head(1000).to_csv("testDataForLibra.csv")

In [ ]:
from libra import client
a_client = client('testDataForLibra.csv')

In [ ]:
a_client.neural_network_query('Identify Drivetrain', epochs=10, drop=['ID', 'Unnamed: 0'], save_model=True)

In [ ]:
# a_client.models['classification_ANN'].plots() #access plots
a_client.analyze()

In [ ]:
DataFrameOfCarData.tail(1000).to_csv("OOStestDataForLibra.csv")

In [ ]:
new_data = pd.read_csv('OOStestDataForLibra.csv')
predictionsOnOOSdata = a_client.predict(new_data.drop(['Drivetrain'], axis=1), model='classification_ANN')

In [ ]:
new_data['Predictions'] = predictionsOnOOSdata

In [ ]:
new_data.to_csv("new_dataPredictions.csv")